# 4. Future Overpass Forecast

Predicts upcoming satellite overpasses of your AOI, so you can plan field
data collection to align with them.

**How it works**: Sentinel-1/2, Landsat, and NISAR fly fixed repeat-orbit
paths, so the same ground track returns on a predictable cycle. Rather
than assuming the textbook nominal cycle (10 days/satellite for
Sentinel-2, 12 for Sentinel-1 and NISAR, 16 for Landsat), this notebook
looks at the last ~60 days of *actual* acquisitions over your specific
AOI, detects the real day-to-day gap pattern per satellite (which can
differ from the nominal figure — e.g. an AOI sitting in the overlap zone
between two adjacent orbit paths gets imaged more often than the nominal
cycle implies), and projects that pattern forward. MODIS (Terra/Aqua) is
a near-daily systematic global product, so its pattern is typically a
flat 1-day cycle.

**Sensors covered**: Sentinel-1, Sentinel-2, Landsat 8/9, NISAR, and
MODIS Terra/Aqua. **Sentinel-6 is excluded** — it's an ocean-only
altimetry mission, so a land AOI never gets a real overpass to build a
forecast from.

**Important caveats**:
- **Sentinel-2, Landsat, and NISAR** are acquired systematically over
  land on every overpass, so these predictions are reliable.
- **MODIS** (Terra/Aqua) is a daily systematic global product, so its
  forecast is effectively "tomorrow, and every day after" — reliable,
  but not a very selective planning signal on its own.
- **Sentinel-1** acquisition is *tasked* by ESA according to an evolving
  observation plan — the satellite passes over on a predictable orbital
  schedule, but whether it actually collects SAR data for your specific
  AOI on a given pass can vary and isn't guaranteed by orbital mechanics
  alone. Treat Sentinel-1 predictions as lower-confidence, especially if
  there's too little recent history to detect a real pattern (falls back
  to the nominal cycle in that case, flagged in the `basis` column).
- **NISAR** is still in its early-mission/commissioning phase (provisional
  data), so its historical acquisition pattern may not yet reflect the
  mission's eventual steady-state observation plan.
- This predicts **when the satellite will be overhead**, not whether the
  resulting image will be cloud-free — cloud cover can't be known in
  advance.
- Predicted time-of-day is carried from the historical pattern and is
  typically accurate to within a few minutes, not exact.

**Requires**: run `1_AOI_Selection.ipynb` first (only needs the AOI, not
the search/download notebooks).

## Setup \u2014 authenticate and initialize Earth Engine

In [1]:
import ee

EE_PROJECT = "rosy-precinct-498822-e1"  # <-- your GEE-enabled Cloud project

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")

Earth Engine initialized.


## Load your AOI

In [2]:
from pathlib import Path

import pandas as pd

from aoi_export import load_geometry

AOI_NAME = "my_aoi"
OUTPUT_DIR = "output"

geojson_path = Path(OUTPUT_DIR) / f"{AOI_NAME}.geojson"
if not geojson_path.exists():
    raise FileNotFoundError(f"{geojson_path} not found \u2014 run 1_AOI_Selection.ipynb first.")

geometry = load_geometry(geojson_path)
print("AOI area (km^2):", geometry.area(1).divide(1e6).getInfo())

AOI area (km^2): 17.253048761760592


## Forecast configuration

In [3]:
from datetime import date, timedelta

TODAY = date.today()
HISTORY_LOOKBACK_DAYS = 60   # how far back to look for the real acquisition pattern
FORECAST_HORIZON_DAYS = 90   # how far into the future to predict

HISTORY_START = (TODAY - timedelta(days=HISTORY_LOOKBACK_DAYS)).isoformat()
HISTORY_END = (TODAY + timedelta(days=1)).isoformat()  # EE/CMR date filters are end-exclusive

print(f"Today: {TODAY}")
print(f"History window (to detect the pattern): {HISTORY_START} to {TODAY}")
print(f"Forecast horizon: through {TODAY + timedelta(days=FORECAST_HORIZON_DAYS)}")

Today: 2026-08-24
History window (to detect the pattern): 2026-06-25 to 2026-08-24
Forecast horizon: through 2026-11-22


## Gather recent overpass history

One row per historical acquisition, with `platform`, `id`, and `time`.
Each pass's real footprint geometry is saved into a separate `FOOTPRINTS`
dict (keyed by `(platform, id)`) rather than as a DataFrame column, so the
printed table stays readable — it's used only by the map section below.

In [4]:
import requests

FOOTPRINTS = {}  # (platform, id) -> GeoJSON geometry, used only by the map below
rows = []

# Sentinel-2 via Earth Engine (SPACECRAFT_NAME gives "Sentinel-2A"/"2B"/"2C" directly)
s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(geometry).filterDate(HISTORY_START, HISTORY_END)


def _s2_props(img):
    return ee.Feature(
        img.geometry(),
        {
            "id": img.get("system:index"),
            "platform": img.get("SPACECRAFT_NAME"),
            "time": img.date().format("YYYY-MM-dd'T'HH:mm:ss"),
        },
    )


info = ee.FeatureCollection(s2_col.map(_s2_props)).getInfo()
print(f"Sentinel-2: {len(info['features'])} historical scene(s)")
for f in info["features"]:
    props = f["properties"]
    FOOTPRINTS[(props["platform"], props["id"])] = f["geometry"]
    rows.append(props)

# Landsat 8/9 via Earth Engine (SPACECRAFT_ID gives "LANDSAT_8"/"LANDSAT_9")
for collection_id in ["LANDSAT/LC08/C02/T1_L2", "LANDSAT/LC09/C02/T1_L2"]:
    col = ee.ImageCollection(collection_id).filterBounds(geometry).filterDate(HISTORY_START, HISTORY_END)

    def _ls_props(img):
        return ee.Feature(
            img.geometry(),
            {
                "id": img.get("system:index"),
                "platform": img.get("SPACECRAFT_ID"),
                "time": img.date().format("YYYY-MM-dd'T'HH:mm:ss"),
            },
        )

    info = ee.FeatureCollection(col.map(_ls_props)).getInfo()
    print(f"{collection_id}: {len(info['features'])} historical scene(s)")
    for f in info["features"]:
        props = f["properties"]
        FOOTPRINTS[(props["platform"], props["id"])] = f["geometry"]
        rows.append(props)

# MODIS Terra/Aqua Snow Cover via Earth Engine — daily global product, so its
# historical pattern is the daily overpass cadence, not the 16-day VI composite cycle.
for collection_id, platform in [("MODIS/061/MOD10A1", "MODIS Terra"), ("MODIS/061/MYD10A1", "MODIS Aqua")]:
    col = ee.ImageCollection(collection_id).filterBounds(geometry).filterDate(HISTORY_START, HISTORY_END)

    def _modis_props(img):
        return ee.Feature(
            img.geometry(),
            {
                "id": img.get("system:index"),
                "time": img.date().format("YYYY-MM-dd'T'HH:mm:ss"),
            },
        )

    info = ee.FeatureCollection(col.map(_modis_props)).getInfo()
    print(f"{platform}: {len(info['features'])} historical scene(s)")
    for f in info["features"]:
        props = f["properties"]
        props["platform"] = platform
        FOOTPRINTS[(platform, props["id"])] = f["geometry"]
        rows.append(props)

# Sentinel-1 GRD and NISAR L2 GCOV via ASF DAAC / CMR (not in Earth Engine's catalog)
coords = geometry.bounds(1).coordinates().getInfo()[0]
lons = [c[0] for c in coords]
lats = [c[1] for c in coords]
bbox = f"{min(lons)},{min(lats)},{max(lons)},{max(lats)}"


def _fetch_asf_history(short_names, platform_from_id):
    """CMR granule search against ASF DAAC; platform_from_id(granule_id) -> platform label."""
    resp = requests.get(
        "https://cmr.earthdata.nasa.gov/search/granules.json",
        params={
            "short_name": short_names,
            "provider": "ASF",
            "bounding_box": bbox,
            "temporal": f"{HISTORY_START}T00:00:00Z,{HISTORY_END}T00:00:00Z",
            "page_size": 200,
        },
        timeout=60,
    )
    entries = resp.json()["feed"]["entry"]
    for g in entries:
        granule_id = g.get("producer_granule_id", "")
        platform = platform_from_id(granule_id)
        if not platform:
            continue
        polygons = g.get("polygons")
        if polygons:
            # CMR polygon strings are "lat lon lat lon ..."; swap to (lon, lat) for GeoJSON.
            ring = polygons[0][0].split()
            points = [[float(ring[i + 1]), float(ring[i])] for i in range(0, len(ring), 2)]
            FOOTPRINTS[(platform, granule_id)] = {"type": "Polygon", "coordinates": [points]}
        rows.append({"id": granule_id, "platform": platform, "time": g.get("time_start")})
    return len(entries)


def _s1_platform(granule_id):
    return f"Sentinel-1{granule_id[2]}" if granule_id.startswith("S1") else None


n = _fetch_asf_history(
    ["SENTINEL-1A_DP_GRD_HIGH", "SENTINEL-1B_DP_GRD_HIGH", "SENTINEL-1C_DP_GRD_HIGH"], _s1_platform
)
print(f"Sentinel-1 GRD: {n} historical scene(s)")

n = _fetch_asf_history(["NISAR_L2_GCOV_PROVISIONAL_V1"], lambda granule_id: "NISAR")
print(f"NISAR L2 GCOV: {n} historical scene(s)")

history_df = pd.DataFrame(rows)
history_df["time"] = pd.to_datetime(history_df["time"], format="ISO8601", utc=True).dt.tz_localize(None)
history_df = history_df.sort_values("time").reset_index(drop=True)
print(f"\n{len(history_df)} total historical overpasses across {history_df['platform'].nunique()} platform(s).")
history_df

Sentinel-2: 30 historical scene(s)


LANDSAT/LC08/C02/T1_L2: 6 historical scene(s)
LANDSAT/LC09/C02/T1_L2: 7 historical scene(s)


MODIS Terra: 58 historical scene(s)


MODIS Aqua: 58 historical scene(s)


Sentinel-1 GRD: 0 historical scene(s)


NISAR L2 GCOV: 9 historical scene(s)

168 total historical overpasses across 8 platform(s).


,id,platform,time
0,2026_06_25,MODIS Terra,2026-06-25 00:00:00
1,2026_06_25,MODIS Aqua,2026-06-25 00:00:00
2,NISAR_L2_PR_GCOV_023_156_D_072_4005_DHDH_A_202...,NISAR,2026-06-25 00:44:39
3,NISAR_L2_PR_GCOV_023_156_D_072_4005_DHDH_A_202...,NISAR,2026-06-25 00:44:39
4,20260625T162829_20260625T163545_T16SCC,Sentinel-2B,2026-06-25 16:44:11
...,...,...,...
163,2026_08_20,MODIS Terra,2026-08-20 00:00:00
164,2026_08_21,MODIS Terra,2026-08-21 00:00:00
165,20260821T163701_20260821T163703_T16SCC,Sentinel-2A,2026-08-21 16:44:27
166,2026_08_22,MODIS Aqua,2026-08-22 00:00:00


## Detect each platform's real revisit pattern and project it forward

For each satellite (Sentinel-1A/B/C, Sentinel-2A/B/C, Landsat 8/9, NISAR,
MODIS Terra/Aqua), finds the smallest repeating gap pattern in its recent
history (e.g. an alternating `[6, 13]`-day pattern, or a flat `[8]`), and
continues that pattern from the most recent observation. Falls back to the
textbook nominal cycle when there's too little history to detect a real
pattern (flagged in `basis`).

In [5]:
NOMINAL_REPEAT_CYCLE_DAYS = {
    "Sentinel-2A": 10, "Sentinel-2B": 10, "Sentinel-2C": 10,
    "Sentinel-1A": 12, "Sentinel-1B": 12, "Sentinel-1C": 12,
    "LANDSAT_8": 16, "LANDSAT_9": 16,
    "NISAR": 12,
    "MODIS Terra": 1, "MODIS Aqua": 1,
}


def _detect_period(gaps, max_period=4):
    """Smallest period P such that the last P gaps match the P gaps before them (±1 day)."""
    for period in range(1, min(max_period, len(gaps) // 2) + 1):
        recent = gaps[-period:]
        prior = gaps[-2 * period : -period]
        if len(prior) == period and all(abs(a - b) <= 1 for a, b in zip(recent, prior)):
            return period
    return None


forecast_rows = []
for platform, group in history_df.groupby("platform"):
    nominal_cycle = NOMINAL_REPEAT_CYCLE_DAYS.get(platform)
    if nominal_cycle is None:
        print(f"WARNING: no known nominal cycle for platform {platform!r}, skipping")
        continue

    dates = sorted(group["time"].tolist())
    gaps = [(dates[i + 1] - dates[i]).days for i in range(len(dates) - 1)]
    period = _detect_period(gaps) if len(gaps) >= 2 else None

    last_seen = dates[-1]
    if period:
        pattern = gaps[-period:]
        basis = f"empirical pattern {pattern} (from {len(dates)} recent obs.)"
    else:
        pattern = [nominal_cycle]
        basis = f"nominal {nominal_cycle}-day cycle (only {len(dates)} recent obs. — pattern not confirmed)"

    current = last_seen
    i = 0
    while True:
        gap = pattern[i % len(pattern)]
        current = current + timedelta(days=gap)
        if current.date() > TODAY + timedelta(days=FORECAST_HORIZON_DAYS):
            break
        if current.date() >= TODAY:
            forecast_rows.append(
                {
                    "platform": platform,
                    "predicted_datetime_utc": current,
                    "days_until": (current.date() - TODAY).days,
                    "basis": basis,
                    "last_observed": last_seen,
                }
            )
        i += 1

forecast_df = pd.DataFrame(forecast_rows).sort_values("predicted_datetime_utc").reset_index(drop=True)
print(f"{len(forecast_df)} predicted overpass(es) through {TODAY + timedelta(days=FORECAST_HORIZON_DAYS)}.")
forecast_df

218 predicted overpass(es) through 2026-11-22.


,platform,predicted_datetime_utc,days_until,basis,last_observed
0,MODIS Aqua,2026-08-24 00:00:00,0,empirical pattern [2] (from 58 recent obs.),2026-08-22 00:00:00
1,MODIS Terra,2026-08-24 00:00:00,0,empirical pattern [1] (from 58 recent obs.),2026-08-21 00:00:00
2,MODIS Terra,2026-08-25 00:00:00,1,empirical pattern [1] (from 58 recent obs.),2026-08-21 00:00:00
3,NISAR,2026-08-25 11:11:17,1,nominal 12-day cycle (only 9 recent obs. — pat...,2026-08-13 11:11:17
4,LANDSAT_8,2026-08-25 16:24:47,1,empirical pattern [8] (from 6 recent obs.),2026-08-09 16:24:47
...,...,...,...,...,...
213,LANDSAT_8,2026-11-21 16:24:47,89,empirical pattern [8] (from 6 recent obs.),2026-08-09 16:24:47
214,LANDSAT_9,2026-11-21 16:24:56,89,empirical pattern [8] (from 7 recent obs.),2026-08-17 16:24:56
215,Sentinel-2B,2026-11-21 16:54:06,89,"empirical pattern [6, 3] (from 12 recent obs.)",2026-08-17 16:54:06
216,MODIS Terra,2026-11-22 00:00:00,90,empirical pattern [1] (from 58 recent obs.),2026-08-21 00:00:00


## Export the forecast

In [6]:
out_csv = Path(OUTPUT_DIR) / f"{AOI_NAME}_overpass_forecast.csv"
forecast_df.to_csv(out_csv, index=False)
print(f"Saved {len(forecast_df)} predicted overpasses -> {out_csv.resolve()}")

Saved 218 predicted overpasses -> C:\Users\Say70\OneDrive - Mississippi State University\Desktop\Satellite Data Search\output\my_aoi_overpass_forecast.csv


## Visualize recent overpass footprints on the map

Draws the AOI and every historical pass's real swath footprint (from the
last `HISTORY_LOOKBACK_DAYS` days) as colored outlines — one color per
satellite platform, with a legend — so you can see exactly how each
platform has actually been covering your AOI, not just abstract dates.

In [7]:
import geemap

AOI_COLOR = "#ef4444"
PLATFORM_COLORS = {
    "Sentinel-2A": "#eab308", "Sentinel-2B": "#facc15", "Sentinel-2C": "#fde047",
    "Sentinel-1A": "#14b8a6", "Sentinel-1B": "#0d9488", "Sentinel-1C": "#0f766e",
    "LANDSAT_8": "#f97316", "LANDSAT_9": "#c2410c",
    "NISAR": "#a855f7",
    "MODIS Terra": "#0ea5e9", "MODIS Aqua": "#38bdf8",
}


def _outline(fc_or_geom, color, width):
    """Render a geometry/FeatureCollection as a boundary-only (unfilled) image layer."""
    fc = fc_or_geom if isinstance(fc_or_geom, ee.FeatureCollection) else ee.FeatureCollection([ee.Feature(fc_or_geom)])
    return ee.Image().byte().paint(fc, 1, width).visualize(palette=[color])


overpass_map = geemap.Map()
overpass_map.add_basemap("Esri.WorldImagery")
overpass_map.addLayer(_outline(geometry, AOI_COLOR, 3), {}, "AOI")

legend_dict = {"AOI": AOI_COLOR}
for platform, color in PLATFORM_COLORS.items():
    subset = history_df[history_df["platform"] == platform]
    features = []
    for _, row in subset.iterrows():
        footprint = FOOTPRINTS.get((row["platform"], row["id"]))
        if footprint:
            features.append(ee.Feature(ee.Geometry(footprint), {"id": row["id"]}))
    if not features:
        continue
    fc = ee.FeatureCollection(features)
    overpass_map.addLayer(_outline(fc, color, 2), {}, f"{platform} ({len(features)} passes)")
    legend_dict[f"{platform} ({len(features)})"] = color

overpass_map.add_legend(title="Recent overpasses", legend_dict=legend_dict)
overpass_map.centerObject(geometry, zoom=9)
overpass_map

Map(center=[33.47402537810104, -88.77396550000127], controls=(WidgetControl(options=['position', 'transparent_…